# 12 — Capstone: Conditional Generator

End-to-end latent diffusion pipeline with CFG, LoRA fine-tuning, and evaluation.

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm

torch.manual_seed(42)
np.random.seed(42)

# Dataset: 4 classes in 6D, compress to 2D via VAE, then conditional diffusion
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=3000, n_features=6, n_informative=5,
                            n_classes=4, n_clusters_per_class=1, random_state=42)
X = StandardScaler().fit_transform(X)
X_data = torch.tensor(X, dtype=torch.float32)
y_data = torch.tensor(y, dtype=torch.long)

N_CLASSES = 4
N_FEAT = 6
LATENT_DIM = 3

# Stage 1: VAE compressor
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(N_FEAT, 32), nn.ReLU())
        self.mu_head = nn.Linear(32, LATENT_DIM)
        self.lv_head = nn.Linear(32, LATENT_DIM)
        self.dec = nn.Sequential(nn.Linear(LATENT_DIM, 32), nn.ReLU(), nn.Linear(32, N_FEAT))

    def encode(self, x):
        h = self.enc(x)
        return self.mu_head(h), self.lv_head(h)

    def decode(self, z): return self.dec(z)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5*lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv

vae = VAE()
opt_v = torch.optim.Adam(vae.parameters(), lr=3e-3)
for _ in range(500):
    r, mu, lv = vae(X_data)
    loss = ((r - X_data)**2).mean() + 0.05 * (-0.5*(1+lv-mu**2-lv.exp())).mean()
    opt_v.zero_grad(); loss.backward(); opt_v.step()

vae.eval()
with torch.no_grad():
    z_data, _ = vae.encode(X_data)
print(f'VAE trained. Latent shape: {z_data.shape}')

In [ ]:
# Stage 2: Conditional latent diffusion with CFG
T = 100
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
NULL = N_CLASSES

class ConditionalLatentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.time_emb = nn.Embedding(T, 32)
        self.class_emb = nn.Embedding(N_CLASSES+1, 32)
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM+64, 128), nn.GELU(),
            nn.Linear(128, 64), nn.GELU(),
            nn.Linear(64, LATENT_DIM)
        )
    def forward(self, z, t, c):
        h = torch.cat([z, self.time_emb(t), self.class_emb(c)], dim=-1)
        return self.net(h)

cond_net = ConditionalLatentNet()
opt_c = torch.optim.Adam(cond_net.parameters(), lr=3e-4)

for step in range(3000):
    idx = torch.randint(0, len(z_data), (256,))
    z0 = z_data[idx].detach()
    c = y_data[idx].clone()
    t = torch.randint(0, T, (256,))
    mask = torch.rand(256) < 0.15
    c[mask] = NULL
    noise = torch.randn_like(z0)
    zt = alpha_bars[t].sqrt().unsqueeze(1)*z0 + (1-alpha_bars[t]).sqrt().unsqueeze(1)*noise
    loss = ((cond_net(zt, t, c) - noise)**2).mean()
    opt_c.zero_grad(); loss.backward(); opt_c.step()

print('Conditional latent diffusion trained')

In [ ]:
# Stage 3: LoRA fine-tune on one class
# Freeze and add LoRA to second layer
for p in cond_net.parameters(): p.requires_grad_(False)

# LoRA matrices for the second linear
rank = 4
lora_A = nn.Parameter(torch.randn(rank, 128) * 0.01)
lora_B = nn.Parameter(torch.zeros(64, rank))
scale = 8.0 / rank

opt_lora = torch.optim.Adam([lora_A, lora_B], lr=1e-3)

# Fine-tune on class 0 data only
class0_mask = (y_data == 0)
z_class0 = z_data[class0_mask].detach()

for step in range(500):
    idx = torch.randint(0, len(z_class0), (64,))
    z0 = z_class0[idx]
    c = torch.zeros(64, dtype=torch.long)
    t = torch.randint(0, T, (64,))
    noise = torch.randn_like(z0)
    zt = alpha_bars[t].sqrt().unsqueeze(1)*z0 + (1-alpha_bars[t]).sqrt().unsqueeze(1)*noise

    # Forward with LoRA
    h = torch.cat([zt, cond_net.time_emb(t), cond_net.class_emb(c)], dim=-1)
    h1 = torch.nn.functional.gelu(cond_net.net[0](h))
    h2 = torch.nn.functional.gelu(cond_net.net[1](h1) + (h1 @ lora_A.T @ lora_B.T) * scale)
    pred = cond_net.net[2](h2)

    loss = ((pred - noise)**2).mean()
    opt_lora.zero_grad(); loss.backward(); opt_lora.step()

lora_params = lora_A.numel() + lora_B.numel()
total_params = sum(p.numel() for p in cond_net.parameters())
print(f'LoRA fine-tuning done. LoRA params: {lora_params} / {total_params}')

In [ ]:
# Stage 4: Evaluate with FID + Precision/Recall
@torch.no_grad()
def sample_conditional(model, class_label, n_samples, guidance=2.0):
    z = torch.randn(n_samples, LATENT_DIM)
    c_c = torch.full((n_samples,), class_label, dtype=torch.long)
    c_u = torch.full((n_samples,), NULL, dtype=torch.long)
    for t_val in range(T-1, -1, -1):
        t_b = torch.full((n_samples,), t_val, dtype=torch.long)
        eps_c = model(z, t_b, c_c)
        eps_u = model(z, t_b, c_u)
        eps = (1+guidance)*eps_c - guidance*eps_u
        mean = (1/alphas[t_val].sqrt())*(z - betas[t_val]/(1-alpha_bars[t_val]).sqrt()*eps)
        if t_val > 0:
            z = mean + betas[t_val].sqrt()*torch.randn_like(z)
        else:
            z = mean
    return z

# FID in latent space per class
from scipy.linalg import sqrtm
def fid_latent(real_z, fake_z):
    mu_r, mu_g = real_z.mean(0), fake_z.mean(0)
    sr = np.cov(real_z, rowvar=False) + np.eye(LATENT_DIM)*1e-6
    sg = np.cov(fake_z, rowvar=False) + np.eye(LATENT_DIM)*1e-6
    cm, _ = sqrtm(sr @ sg, disp=False)
    if np.iscomplexobj(cm): cm = cm.real
    return float(np.real((mu_r-mu_g)@(mu_r-mu_g) + np.trace(sr + sg - 2*cm)))

print('Evaluation per class (latent FID):')
for cls in range(N_CLASSES):
    real_z = z_data[y_data == cls][:100].detach().numpy()
    fake_z = sample_conditional(cond_net, cls, 100).numpy()
    fid = fid_latent(real_z, fake_z)
    print(f'  Class {cls}: FID = {fid:.3f}')

print('\nSeries 23 (Generative Models & Diffusion) complete.')